# Scraping test

#### Prueba de scraping de una noticia en particular en distintos medios con distintas vistas

Importamos las librerias necesarias

In [15]:
import requests
import time
from bs4 import BeautifulSoup
from GoogleNews import GoogleNews

Le pedimos al usuario que ingrese la noticia a buscar y le solicitamos a la libreria que lo busque en el navegador

In [16]:
query = input("Tema a buscar: ")

googlenews = GoogleNews(lang='es', region='AR')
googlenews.search(query)

resultados = []

for page in range(1, 30):
    googlenews.getpage(page)
    resultados.extend(googlenews.result())

Podemos ver las noticias que recolecto corriendo el siguiente codigo:

In [17]:
for n in resultados:
    print(f" {n['title']}")
    print(f" {n['media']}")
    print(f" {n['link']}\n")

Seteamos una lista de medios de comunicacion objetivo que se pueda expandir a futuro

In [18]:
medios_objetivo = [
    "clarin.com",
    "a24.com",
    "pagina12.com.ar"
]

Establecemos metodos de filtrado inteligente para no traer noticias que sean poco relevantes:

In [ ]:
from urllib.parse import urlparse

def dominio_de_url(url):
    """Devuelve el dominio principal del link, por ejemplo 'clarin.com'."""
    return urlparse(url).netloc.replace('www.', '')

def es_valida(noticia):
    """Filtra noticias que no deberían considerarse."""
    titulo = noticia.get("title", "").strip().lower()
    link = noticia.get("link", "")
    dominio = dominio_de_url(link)

    if dominio not in medios_objetivo:
        return False

    if dominio == "clarin.com" and titulo.startswith("video"):
        return False

    if not link or not titulo:
        return False

    return True

def elegir_mas_relevante_por_medio(noticias, consulta):
    mejores = {}
    consulta = consulta.lower()

    for noticia in noticias:
        # if not es_valida(noticia):
        #   continue
        
        link = noticia.get("link", "")
        dominio = dominio_de_url(link)
        titulo = noticia.get("title", "").lower()

        # Medida de relevancia simple: cuántas palabras de la consulta aparecen en el título
        relevancia = sum(palabra in titulo for palabra in consulta.split())

        # Guardamos la de mayor relevancia por dominio
        if dominio not in mejores or relevancia > mejores[dominio]["relevancia"]:
            mejores[dominio] = {"noticia": noticia, "relevancia": relevancia}

    # Devolvemos solo las noticias elegidas
    return [data["noticia"] for data in mejores.values()]

Ejectuamos una consulta de prueba

In [21]:
noticias_principales = elegir_mas_relevante_por_medio(resultados, query)

for n in noticias_principales:
    print(f"  {n['title']}")
    print(f"  {n['media']}")
    print(f"  {n['link']}\n")


In [22]:
print(noticias_principales)

[]
